# VoxCPM Text-to-Voice (Colab)

Ready-to-run notebook for Reelday.ph marketing voiceover using [VoxCPM](https://github.com/OpenBMB/VoxCPM) (Apache-2.0, free for commercial use).

**Before you run:** set the GPU runtime.
`Runtime` → `Change runtime type` → Hardware accelerator = **T4 GPU** → Save.

Then run the cells top to bottom. Cell 1 (install) takes ~3–5 min on a fresh session and must be re-run each new session.

## 1. Check GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print('CUDA available:', torch.cuda.is_available())

## 2. Install VoxCPM (~3–5 min, re-run each session)

In [ ]:
!pip install -q voxcpm soundfile
print('Done. If you see dependency warnings, they are usually safe to ignore.')

## 3. Load the model (downloads weights on first run)

In [ ]:
from voxcpm import VoxCPM
import soundfile as sf
from IPython.display import Audio, display

model = VoxCPM.from_pretrained(
    "openbmb/VoxCPM2",
    load_denoiser=False,
)
print('Model loaded. Sample rate:', model.tts_model.sample_rate)

## 4. Generate voiceover

Edit `TEXT` below, run the cell, then play / download the result.
- `cfg_value`: higher = sticks closer to a reference style (2.0 is a good default).
- `inference_timesteps`: higher = better quality but slower (10 is a good default).

In [ ]:
TEXT = "Capture every moment of your big day with Reelday. Real videographers, edited reels, delivered fast."
OUTPUT_FILE = "voiceover.wav"

wav = model.generate(
    text=TEXT,
    cfg_value=2.0,
    inference_timesteps=10,
)
sf.write(OUTPUT_FILE, wav, model.tts_model.sample_rate)
print('Saved', OUTPUT_FILE)
display(Audio(OUTPUT_FILE))

## 5. Download the audio file

In [ ]:
from google.colab import files
files.download(OUTPUT_FILE)

### Batch generation (optional)

Generate several clips at once — useful for multiple ad variations.

In [ ]:
LINES = {
    "hook_1": "Your wedding deserves more than photos.",
    "hook_2": "From I do to the dance floor — we film it all.",
    "cta":    "Book your Reelday today. It's free to start.",
}

for name, line in LINES.items():
    wav = model.generate(text=line, cfg_value=2.0, inference_timesteps=10)
    fname = f"{name}.wav"
    sf.write(fname, wav, model.tts_model.sample_rate)
    print(name)
    display(Audio(fname))